# Implementing Explainable AI Techniques (SHAP, LIME)

## 📚 Learning Objectives

By completing this notebook, you will:
- Implement SHAP for model explanation
- Implement LIME for local explanations
- Explain model predictions
- Interpret feature importance
- Apply explainability to AI models

## 🔗 Prerequisites

- ✅ Understanding of model interpretability
- ✅ Understanding of explainable AI
- ✅ SHAP/LIME knowledge

---

This notebook covers practical activities from **Course 06, Unit 4**:
- Implementing explainable AI techniques (SHAP, LIME)

---

## Introduction

**Explainable AI (XAI)** techniques like SHAP and LIME provide insights into model decisions, enabling transparency and accountability in AI systems.

## 📥 Inputs & 📤 Outputs

**Inputs:** What we use in this notebook

- Libraries and concepts as introduced in this notebook; see prerequisites and code comments.

**Outputs:** What you'll see when you run the cells

- Printed results, figures, and summaries as shown when you run the cells.

---

In [1]:
import shap
import lime
import numpy as np

print("✅ Libraries imported!")
print("\nExplainable AI Techniques")
print("=" * 60)

print("\nSHAP (SHapley Additive exPlanations):")
print(" - Game theory-based")
print(" - Feature importance")
print(" - Global and local explanations")
print(" - Consistent explanations")

print("\nLIME (Local Interpretable Model-agnostic Explanations):")
print(" - Local explanations")
print(" - Model-agnostic")
print(" - Perturbation-based")
print(" - Interpretable models")

print("\nApplications:")
print(" - Model debugging")
print(" - Feature importance")
print(" - Regulatory compliance")
print(" - User trust")

print("\n✅ Explainable AI concepts understood!")

✅ Libraries imported!

Explainable AI Techniques

SHAP (SHapley Additive exPlanations):
 - Game theory-based
 - Feature importance
 - Global and local explanations
 - Consistent explanations

LIME (Local Interpretable Model-agnostic Explanations):
 - Local explanations
 - Model-agnostic
 - Perturbation-based
 - Interpretable models

Applications:
 - Model debugging
 - Feature importance
 - Regulatory compliance
 - User trust

✅ Explainable AI concepts understood!


In [2]:
# Practice: compute BOTH SHAP and LIME explanations on one model
import numpy as np
import pandas as pd
import shap
from lime.lime_tabular import LimeTabularExplainer
import warnings
warnings.filterwarnings('ignore')  # silence sklearn feature-name warning from LIME's numpy inputs
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

np.random.seed(42)
n = 600
X = pd.DataFrame({
    'income':         np.random.normal(55, 18, n).clip(10, 150),
    'debt_ratio':     np.random.uniform(0.05, 0.6, n),
    'years_employed': np.random.randint(0, 25, n),
    'credit_history': np.random.normal(650, 80, n).clip(350, 850),
})
score = (0.03*(X['income']-55) - 4.0*(X['debt_ratio']-0.3)
         + 0.05*X['years_employed'] + 0.008*(X['credit_history']-650))
y = (score + np.random.normal(0, 0.35, n) > 0).astype(int)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25,
                                                    random_state=42, stratify=y)
model = RandomForestClassifier(n_estimators=120, random_state=42).fit(X_train, y_train)
print(f"Model test accuracy: {accuracy_score(y_test, model.predict(X_test)):.3f}")

# ---- SHAP ----
sv = shap.TreeExplainer(model).shap_values(X_test)
sv1 = sv[1] if isinstance(sv, list) else (sv[:, :, 1] if sv.ndim == 3 else sv)
shap_rank = pd.Series(np.abs(sv1).mean(axis=0), index=X_test.columns
                      ).sort_values(ascending=False)
print("\nSHAP global importance (mean |SHAP|):")
for f, v in shap_rank.items():
    print(f"  {f:<15} {v:.4f}")

# ---- LIME (instance 0) ----
lime_exp = LimeTabularExplainer(
    X_train.values, feature_names=list(X_train.columns),
    class_names=['denied', 'approved'], mode='classification', random_state=42,
).explain_instance(X_test.iloc[0].values, model.predict_proba, num_features=4)
print(f"\nLIME explanation for applicant #0 "
      f"(approval prob {model.predict_proba(X_test.iloc[[0]])[0, 1]:.2f}):")
for rule, w in lime_exp.as_list():
    print(f"  {rule:<35} {w:+.4f}")

# ---- SHAP local, same instance, for comparison ----
shap_local = pd.Series(sv1[0], index=X_test.columns)
print("\nSHAP local attributions for the SAME applicant:")
for f, v in shap_local.sort_values(key=abs, ascending=False).items():
    print(f"  {f:<15} {v:+.4f}")

agree = shap_rank.index[0]
print(f"\nBoth methods computed real explanations from the model. Compare the")
print(f"local views above: do SHAP and LIME point at the same features for")
print(f"applicant #0? Globally, SHAP ranks '{agree}' as most influential.")


Model test accuracy: 0.887

SHAP global importance (mean |SHAP|):
  debt_ratio      0.2085
  credit_history  0.1569
  income          0.1335
  years_employed  0.0714

LIME explanation for applicant #0 (approval prob 0.97):
  income > 66.04                      +0.2052
  0.17 < debt_ratio <= 0.32           +0.1993
  years_employed <= 5.00              -0.1376
  651.67 < credit_history <= 698.55   +0.1185

SHAP local attributions for the SAME applicant:
  debt_ratio      +0.1656
  credit_history  +0.1240
  income          +0.1172
  years_employed  -0.0771

Both methods computed real explanations from the model. Compare the
local views above: do SHAP and LIME point at the same features for
applicant #0? Globally, SHAP ranks 'debt_ratio' as most influential.
